In [1]:

import sys
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))

from pricing_engine.benchmark import PricingBenchmarkSuite
from pricing_engine.Bandit import ThompsonBandit
from pricing_engine.demand_model import HierarchicalDemandModel
from pricing_engine.safety import SafetyLayer
from pricing_engine.data_loader import load_and_clean_seattle_data





In [2]:
from pathlib import Path



try:
    SCRIPT_DIR = Path(__file__).parent
except NameError:
    SCRIPT_DIR = Path.cwd()

# SCRIPT_DIR is the 'notebooks' folder.
# SCRIPT_DIR.parent is the 'Dynamic Pricing Engine' folder.
PROJECT_ROOT = SCRIPT_DIR.parent

# Now build the path from the project root
CALENDAR_PATH = PROJECT_ROOT / 'data' / 'calendar.csv'
LISTINGS_PATH = PROJECT_ROOT / 'data' / 'listings.csv'



In [3]:
print("Initializing Engine Components...")
df = load_and_clean_seattle_data(CALENDAR_PATH, LISTINGS_PATH)
df['day_of_year'] = df['date'].dt.dayofyear
df['dow'] = df['date'].dt.dayofyear
df['is_weekend'] = (df['dow'] >= 5).astype(int)

hdm = HierarchicalDemandModel(min_obs_for_listing=20)
hdm.fit(df.sample(frac=0.5), ['day_of_year', 'dow', 'is_weekend']) # Train on 50% for speed

agent = ThompsonBandit(hdm, forgetting_factor=0.90)
safety = SafetyLayer()

suite = PricingBenchmarkSuite(agent, safety)
print("✅ Engine Ready for Stress Testing.")


Initializing Engine Components...
✅ Engine Ready for Stress Testing.


## 2. Execute Benchmark Suite

Running Tests B01 through B06...

In [4]:


print("Running B01-B06... (This may take 1-2 minutes)")
df_results = suite.run_all()



Running B01-B06... (This may take 1-2 minutes)


## 3. Readiness Scorecard

Review the Pass/Fail status of each critical invariant.

In [5]:
# Display styled scorecard

def color_status(val):
    color = 'green' if val == 'PASS' else 'red'
    return f'color: {color}; font-weight: bold'

df_results.style.applymap(color_status, subset=['status'])





C:\Users\99sma\AppData\Local\Temp\ipykernel_7156\2028286427.py:7: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  df_results.style.applymap(color_status, subset=['status'])


,test_id,name,status,metrics,message
0,B01,E2E Latency,PASS,{'P99_ms': np.float64(18.99582700119936)},P99: 19.00ms
1,B02,Cold Start,PASS,{},Bounds respected
2,B03,Shock Adaptation,PASS,{'FinalAvg': np.float64(230.15384615384613)},Adapted to 230
3,B04,Causal Sanity,PASS,{'Causal_Beta': np.float64(-2.4938527684624208)},Negative elasticity recovered
4,B05,Adversarial Safety,PASS,{'Violations': 0},Safety layer handled NaN/Inf/Negative inputs correctly
5,B06,Regret Stability,PASS,"{'TotalRegret': 175.0, 'FinalSlope': 0.88}",Regret growth is sub-linear (Agent is learning)


## 4. Detailed Failure Analysis

If any test failed, inspect the metrics below.

In [6]:
for idx, row in df_results.iterrows():
    print(f"--- {row['test_id']}: {row['name']} ---")
    print(f"Status: {row['status']}")
    print(f"Metrics: {row['metrics']}")
    print(f"Log: {row['message']}\n")





--- B01: E2E Latency ---
Status: PASS
Metrics: {'P99_ms': np.float64(18.99582700119936)}
Log: P99: 19.00ms

--- B02: Cold Start ---
Status: PASS
Metrics: {}
Log: Bounds respected

--- B03: Shock Adaptation ---
Status: PASS
Metrics: {'FinalAvg': np.float64(230.15384615384613)}
Log: Adapted to 230

--- B04: Causal Sanity ---
Status: PASS
Metrics: {'Causal_Beta': np.float64(-2.4938527684624208)}
Log: Negative elasticity recovered

--- B05: Adversarial Safety ---
Status: PASS
Metrics: {'Violations': 0}
Log: Safety layer handled NaN/Inf/Negative inputs correctly

--- B06: Regret Stability ---
Status: PASS
Metrics: {'TotalRegret': 175.0, 'FinalSlope': 0.88}
Log: Regret growth is sub-linear (Agent is learning)



## 5. Final GO / NO-GO Decision

Automatic logic to determine deployment eligibility.

In [7]:
failed_tests = df_results[df_results['status'] == 'FAIL']

print("====== FINAL DEPLOYMENT DECISION ======")
if len(failed_tests) == 0:
    print("🚀 GO: All systems nominal. Ready for Shadow Mode.")
else:
    print("🛑 NO-GO: Deployment Blocked.")
print("Blocking Issues:")
for _, row in failed_tests.iterrows():
    print(f" - {row['name']} ({row['message']})")

====== FINAL DEPLOYMENT DECISION ======
🚀 GO: All systems nominal. Ready for Shadow Mode.
Blocking Issues:
